You are provided with a data set titled ‘Lung Cancer Patient Health and Treatment Records’, which captures detailed information about patient demographics, diagnosis stage, lifestyle risk factors, comorbidities and treatment outcomes.
Note: This data set was inspired by various healthcare-related learning resources and is not intended to represent real patient data. 
With the given data set, solve the following tasks using PySpark.

Task 1: Write a function that removes duplicate rows, ensures correct data types for numerical and date columns and converts all ‘yes’/ ‘no’ type fields into 1/0 format.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, DateType
from pyspark.sql.functions import col, when

def clean_data(df):
    """
    Removes duplicates, ensures correct data types, and converts yes/no fields to 1/0.
    """
    # Remove duplicate rows
    df = df.dropDuplicates()
    
    # Convert yes/no columns to 1/0
    yes_no_columns = [field.name for field in df.schema.fields if field.dataType.simpleString() == 'string']
    for column in yes_no_columns:
        df = df.withColumn(column, 
                          when(col(column).isin(['yes', 'Yes', 'YES']), 1)
                          .when(col(column).isin(['no', 'No', 'NO']), 0)
                          .otherwise(col(column)))
    
    # Cast numerical columns to appropriate types
    numerical_columns = ['age', 'bmi', 'treatment_duration_days']
    for column in numerical_columns:
        if column in df.columns:
            df = df.withColumn(column, col(column).cast(DoubleType()))
    
    # Cast date columns to DateType
    date_columns = ['diagnosis_date', 'treatment_end_date']
    for column in date_columns:
        if column in df.columns:
            df = df.withColumn(column, col(column).cast(DateType()))
    
    return df

Task 2: Write a function that adds a new column, treatment_duration_days, which calculates the number of days between the diagnosis and the end of treatment. Then, return the average treatment duration for each treatment type. 

In [ ]:
from pyspark.sql.functions import datediff, avg, col

def treatment_duration_analysis(df):
    """
    Adds treatment_duration_days column and returns average treatment duration by treatment type.
    """
    # Add treatment_duration_days column
    df = df.withColumn('treatment_duration_days', 
                       datediff(col('treatment_end_date'), col('diagnosis_date')))
    
    # Calculate average treatment duration for each treatment type
    avg_duration_by_treatment = df.groupBy('treatment_type').agg(
        avg('treatment_duration_days').alias('avg_treatment_duration')
    )
    
    return avg_duration_by_treatment

Task 3: Write a function that returns the smoking_status group with the highest survival rate.  

In [ ]:
from pyspark.sql.functions import col, sum, count

def highest_survival_smoking_group(df):
    """
    Returns the smoking_status group with the highest survival rate.
    """
    # Calculate survival rate for each smoking_status group
    survival_rates = df.groupBy('smoking_status').agg(
        (sum(col('survived')) / count(col('survived'))).alias('survival_rate')
    )
    
    # Find the group with the highest survival rate
    highest_survival_group = survival_rates.orderBy(col('survival_rate').desc()).first()
    
    return highest_survival_group['smoking_status']

Task 4: Write a function that returns the top three countries with the highest percentage of patients diagnosed in Stage IV.  

In [ ]:
from pyspark.sql.functions import col, sum, count, desc

def top_three_stage_iv_countries(df):
    """
    Returns the top three countries with the highest percentage of patients diagnosed in Stage IV.
    """
    # Calculate total patients and Stage IV patients by country
    stage_iv_percentage = df.groupBy('country').agg(
        (sum(when(col('diagnosis_stage') == 'Stage IV', 1).otherwise(0)) / count(col('country'))).alias('stage_iv_percentage')
    )
    
    # Get top 3 countries with highest Stage IV percentage
    top_three = stage_iv_percentage.orderBy(col('stage_iv_percentage').desc()).limit(3)
    
    return top_three

Task 5: Write a function that filters patients who:  

Are male  

Diagnosed in Stage III or IV  

Have a family history of cancer  

Are current smokers  

Have a BMI > 30  

Survived 

Return the average age and the percentage of these patients who had hypertension.

In [ ]:
from pyspark.sql.functions import col, avg, sum, count, when

def filter_and_analyze_patients(df):
    """
    Filters patients based on criteria and returns average age and hypertension percentage.
    """
    # Filter patients based on all criteria
    filtered_df = df.filter(
        (col('gender') == 'Male') &
        (col('diagnosis_stage').isin(['Stage III', 'Stage IV'])) &
        (col('family_history_cancer') == 1) &
        (col('smoking_status') == 'Current') &
        (col('bmi') > 30) &
        (col('survived') == 1)
    )
    
    # Calculate average age and hypertension percentage
    result = filtered_df.agg(
        avg('age').alias('average_age'),
        (sum(when(col('hypertension') == 1, 1).otherwise(0)) / count('*')).alias('hypertension_percentage')
    )
    
    return result